In [2]:
import pandas as pd
from pyvis.network import Network
import io

# Dados CSV fornecidos
csv_data = """edge_id,case_id,e1_normalizado,e1_text,e1_type,e2_normalizado,e2_text,e2_type,relation_label,intermediate_text
E_001,PMC5137649_01,she,She,PATIENT,contrast,contrast,PROCEDURE,UNDERWENT_PROCEDURE,underwent
E_002,PMC5137649_01,she,She,PATIENT,computed tomography,computed tomography,PROCEDURE,UNDERWENT_PROCEDURE,underwent contrast enhanced
E_003,PMC5137649_01,she,She,PATIENT,a 6cm cystic lesion,a 6cm cystic lesion,VALUE,HAS_VALUE,"underwent contrast enhanced tomography, demonstrating"
E_004,PMC5137649_01,contrast,contrast,PROCEDURE,computed tomography,computed tomography,PROCEDURE,TREATED_BY,enhanced
E_005,PMC5137649_01,contrast,contrast,PROCEDURE,a 6cm cystic lesion,a 6cm cystic lesion,VALUE,HAS_VALUE,"enhanced computed tomography, demonstrating"
E_006,PMC5137649_01,computed tomography,computed tomography,PROCEDURE,a 6cm cystic lesion,a 6cm cystic lesion,VALUE,HAS_VALUE,", demonstrating"
E_007,PMC5137649_01,she,She,PATIENT,eus-fna,EUS-FNA,PROCEDURE,UNDERWENT_PROCEDURE,subsequently underwent
E_008,PMC5137649_01,she,She,PATIENT,normal pancreatic echotexture,normal pancreatic echotexture,FINDING,HAS_SYMPTOM,"subsequently underwent EUS-FNA, which revealed"
E_009,PMC5137649_01,she,She,PATIENT,a cyst,a cyst,DIAGNOSIS,HAS_SYMPTOM,subsequently underwent EUS-FNA pancreatic echotexture and
E_010,PMC5137649_01,eus-fna,EUS-FNA,PROCEDURE,normal pancreatic echotexture,normal pancreatic echotexture,FINDING,SUPPORTS,", which revealed"
E_011,PMC5137649_01,eus-fna,EUS-FNA,PROCEDURE,a cyst,a cyst,DIAGNOSIS,SUPPORTS,", which revealed pancreatic echotexture and"
E_012,PMC5137649_01,normal pancreatic echotexture,normal pancreatic echotexture,FINDING,that,that,PROCEDURE,TREATED_BY,and a cyst x 9cm
E_013,PMC5137649_01,a cyst,a cyst,DIAGNOSIS,that,that,PROCEDURE,TREATED_BY,measuring 6cm x 9cm
E_014,PMC5137649_01,intraoperative endoscopy,Intraoperative endoscopy,PROCEDURE,no mucosal abnormalities,no mucosal abnormalities,FINDING,SUPPORTS,was performed and
E_015,PMC5137649_01,the cyst,the cyst,DIAGNOSIS,the procedure,the procedure,PROCEDURE,TREATED_BY,was adherent to coeliac axis and
E_016,PMC5137649_01,repeat oesophagogastroduodenoscopy,Repeat oesophagogastroduodenoscopy,PROCEDURE,the presence,the presence,PROCEDURE,SUPPORTS,revealed
E_017,PMC5137649_01,she,she,PATIENT,complete resolution,complete resolution,PROCEDURE,UNDERWENT_PROCEDURE,had
E_018,PMC5137649_01,she,she,PATIENT,a return,a return,PROCEDURE,UNDERWENT_PROCEDURE,"had complete resolution nausea, and"
E_019,PMC5137649_01,complete resolution,complete resolution,PROCEDURE,a return,a return,PROCEDURE,TREATED_BY,"of her pain nausea, and"
"""

# Carregar os dados
df = pd.read_csv(io.StringIO(csv_data))

# Inicializar a rede com as configurações de estilo do arquivo de referência
net = Network(height='100vh', width='100%', directed=True, bgcolor='#ffffff', font_color='black')

# Definir as opções de Física e Arestas idênticas ao modelo referenciado
options_string = """
var options = {
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08
    },
    "minVelocity": 0.75,
    "solver": "forceAtlas2Based"
  },
  "edges": {
    "font": {
      "size": 10,
      "align": "middle"
    },
    "arrows": {
      "to": {
        "enabled": true,
        "scaleFactor": 0.5
      }
    }
  }
}
"""
net.set_options(options_string)

# Dicionário de cores inspirado na paleta do arquivo de referência
color_map = {
    'PATIENT': '#79d2a6',     # Verde (referência para o nó 'woman')
    'DIAGNOSIS': '#ff4d4d',   # Vermelho (referência para 'DISEASE')
    'SYMPTOM': '#ff4d4d',     # Vermelho (referência para 'DISEASE')
    'PROCEDURE': '#ffff66',   # Amarelo (referência para 'Procedure/Exam')
    'FINDING': '#66b3ff',     # Azul (referência para 'MedicalConcept')
    'NOT_ENTITY': '#66b3ff',  # Azul (referência para 'MedicalConcept')
    'VALUE': '#d9d9d9'        # Cinza (referência para 'ExamResult')
}

# Adicionar nós e arestas iterando pelo DataFrame
for _, row in df.iterrows():
    
    # Adicionar Nó 1 (Origem) com formato 'dot'
    net.add_node(
        n_id=row['e1_normalizado'], 
        label=row['e1_text'], 
        title=f"Tipo: {row['e1_type']}", 
        color=color_map.get(row['e1_type'], '#cccccc'),
        shape='dot'
    )
    
    # Adicionar Nó 2 (Destino) com formato 'dot'
    net.add_node(
        n_id=row['e2_normalizado'], 
        label=row['e2_text'], 
        title=f"Tipo: {row['e2_type']}", 
        color=color_map.get(row['e2_type'], '#cccccc'),
        shape='dot'
    )
    
    # Adicionar Aresta (Relação)
    net.add_edge(
        source=row['e1_normalizado'], 
        to=row['e2_normalizado'], 
        label=row['relation_label'], 
        title=f"Texto Intermediário: {row['intermediate_text']}",
        color='#808080' # Cor acinzentada padronizada para arestas
    )

# Gerar o arquivo HTML
net.write_html("knowledge_graph_estilizado.html")